# Group 88 - UDL Assignment 2: Part B

## WGAN and WGAN-GP

**Dataset:** CIFAR-10 (pixels scaled to [0, 1])  
**Framework:** PyTorch  
**Members:** Gopikannan G (2024ac05790), Sreejith M V (2024AD05421), Ashwini N (2024ad05029), Lalit Tyagi (2024ac05569), Ashok Pyaram (2024ad05197)

This standalone notebook trains WGAN (weight clipping) and WGAN-GP (gradient penalty), generates 100 samples from each, and compares their recorded evaluation values. Run cells in order on the provisioned GPU environment.


In [ ]:
# Kubeflow environment note: use the preinstalled managed PyTorch environment.
# Do not reinstall NumPy/PyTorch/torchvision while this kernel is running.
# If packages were changed, restart the kernel before running the remaining cells.


In [ ]:
import os
import math
import time
import json
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Preprocessing: normalize CIFAR-10 pixel values to [0, 1].
# This avoids torchvision.ToTensor(), which calls torch.from_numpy and can fail
# when a hosted environment has incompatible PyTorch and NumPy binary builds.
def pil_to_tensor_without_numpy(image):
    image = image.convert('RGB')
    pixels = torch.frombuffer(bytearray(image.tobytes()), dtype=torch.uint8)
    return pixels.view(image.height, image.width, 3).permute(2, 0, 1).float().div(255.0)

transform = transforms.Lambda(pil_to_tensor_without_numpy)

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Single-process loading is robust in hosted notebook environments and makes transform errors easier to diagnose.
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

**Discussion — Setup**

- Confirms training runs on `cuda` (GPU), as required by the assignment's "use provisioned WILP lab infrastructure" instruction — CPU-only execution would make the full sweep (4 $\beta$ values, 3 codebook sizes, 2 GAN variants) impractically slow.
- CIFAR-10 download (170 MB) is a one-time cost; pixel values are loaded via `ToTensor()`, which already normalizes to \[0, 1\] as the assignment's preprocessing step requires — no extra normalization/standardization is applied that would need to be undone before PSNR/FID computation.

In [ ]:
def calculate_psnr(img1, img2):
    """Calculates Mean PSNR between two batches of images [0, 1]"""
    mse = torch.mean((img1 - img2) ** 2, dim=[1, 2, 3])
    mse = torch.clamp(mse, min=1e-8)
    psnr = 20 * torch.log10(1.0 / torch.sqrt(mse))
    return torch.mean(psnr).item()

def calculate_colour_frechet_proxy(real_imgs, gen_imgs, eps=1e-6):
    """Torch-only global-colour Frechet proxy; does not depend on NumPy/SciPy."""
    def features(images):
        return images.mean(dim=(2, 3)) if images.dim() == 4 else images.flatten(1)
    def covariance(values):
        centered = values - values.mean(dim=0, keepdim=True)
        return centered.T @ centered / max(values.size(0) - 1, 1)
    real_features, gen_features = features(real_imgs).float(), features(gen_imgs).float()
    mu_real, mu_gen = real_features.mean(dim=0), gen_features.mean(dim=0)
    sigma_real, sigma_gen = covariance(real_features), covariance(gen_features)
    identity = torch.eye(sigma_real.size(0), device=real_imgs.device, dtype=real_imgs.dtype)
    # sqrt(Sigma_r Sigma_g) is evaluated through a symmetric PSD equivalent.
    eigenvalues, eigenvectors = torch.linalg.eigh(sigma_real + eps * identity)
    sqrt_real = (eigenvectors * eigenvalues.clamp_min(0).sqrt()) @ eigenvectors.T
    middle = sqrt_real @ (sigma_gen + eps * identity) @ sqrt_real
    trace_sqrt = torch.linalg.eigvalsh(middle).clamp_min(0).sqrt().sum()
    fid_score = (mu_real - mu_gen).pow(2).sum() + torch.trace(sigma_real + sigma_gen) - 2 * trace_sqrt
    return fid_score.clamp_min(0).item()

### Part B: WGAN and WGAN-GP [3 Marks]

Train a Wasserstein GAN with weight clipping (WGAN) and its gradient-penalty variant (WGAN-GP) on CIFAR-10, generate random samples, and compare FID scores. Both models share the same Generator/Critic backbone so the comparison isolates the effect of the Lipschitz constraint (clipping vs. gradient penalty).

#### Generator and Critic Architecture

In [ ]:
class Generator(nn.Module):
    """Maps a latent noise vector z -> a 32x32x3 image in [0, 1]."""
    def __init__(self, latent_dim=128, feature_maps=64):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, feature_maps * 4, 4, stride=1, padding=0), # 4x4
            nn.BatchNorm2d(feature_maps * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_maps * 4, feature_maps * 2, 4, stride=2, padding=1), # 8x8
            nn.BatchNorm2d(feature_maps * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_maps * 2, feature_maps, 4, stride=2, padding=1), # 16x16
            nn.BatchNorm2d(feature_maps),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_maps, 3, 4, stride=2, padding=1), # 32x32
            nn.Sigmoid() # Restricts output to [0, 1] to match normalized CIFAR-10 pixels
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


class Critic(nn.Module):
    """Scores an image with an unbounded real-valued score (no sigmoid -> Wasserstein critic, not a classifier)."""
    def __init__(self, feature_maps=64, use_batchnorm=True):
        super(Critic, self).__init__()
        # BatchNorm is disabled for WGAN-GP: batch-level normalization couples samples within a
        # batch, which invalidates the per-sample gradient penalty.
        norm_layer = (lambda c: nn.BatchNorm2d(c)) if use_batchnorm else (lambda c: nn.InstanceNorm2d(c, affine=True))
        self.net = nn.Sequential(
            nn.Conv2d(3, feature_maps, 4, stride=2, padding=1), # 16x16
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_maps, feature_maps * 2, 4, stride=2, padding=1), # 8x8
            norm_layer(feature_maps * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_maps * 2, feature_maps * 4, 4, stride=2, padding=1), # 4x4
            norm_layer(feature_maps * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_maps * 4, 1, 4, stride=1, padding=0) # 1x1 scalar score
        )

    def forward(self, x):
        return self.net(x).view(-1)


def gradient_penalty(critic, real, fake, device):
    """Penalizes deviation of the critic's gradient norm from 1 along lines between real and fake
    samples, enforcing the 1-Lipschitz constraint required by the Wasserstein distance (WGAN-GP)."""
    batch_size = real.size(0)
    eps = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = (eps * real + (1 - eps) * fake).requires_grad_(True)
    scores = critic(interpolated)
    grads = torch.autograd.grad(
        outputs=scores, inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True, retain_graph=True
    )[0]
    grads = grads.view(batch_size, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


**Discussion — Generator/Critic architecture (WGAN & WGAN-GP)**

- Generator mirrors the VAE/VQ-VAE decoders (transposed-conv upsampling 4×4→8×8→16×16→32×32, sigmoid head), keeping output range at \[0, 1\] consistent with the other two models for fair comparison in Part C.
- Critic has **no sigmoid output** — a WGAN critic outputs an unbounded real-valued score approximating the Wasserstein distance, not a real/fake probability like a vanilla GAN discriminator.
- BatchNorm is used for plain WGAN but swapped for InstanceNorm in WGAN-GP, since BatchNorm couples samples within a batch and breaks the per-sample gradient-penalty computation — a deliberate fix from the WGAN-GP paper, not an inconsistency between the two branches.

#### Shared WGAN Training Routine

A single function trains either variant: `clip` for the original WGAN weight-clipping constraint, or `gp` for the WGAN-GP gradient-penalty constraint. The critic is updated `n_critic` times per generator update, matching the original WGAN algorithm.

In [ ]:
def train_wgan(variant, epochs=5, latent_dim=128, n_critic=5, clip_value=0.01, gp_lambda=10.0, lr=5e-5):
    """Trains a WGAN (variant='clip') or WGAN-GP (variant='gp') on CIFAR-10.

    Returns the trained generator plus a PSNR-vs-epoch history (reconstruction proxy: real vs.
    same-batch generated samples) and the final FID against the test set.
    """
    assert variant in {"clip", "gp"}
    run_started = time.perf_counter()
    generator = Generator(latent_dim=latent_dim).to(device)
    critic = Critic(use_batchnorm=(variant == "clip")).to(device)

    if variant == "clip":
        opt_g = torch.optim.RMSprop(generator.parameters(), lr=lr)
        opt_c = torch.optim.RMSprop(critic.parameters(), lr=lr)
    else:
        opt_g = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.5, 0.9))
        opt_c = torch.optim.Adam(critic.parameters(), lr=1e-4, betas=(0.5, 0.9))

    psnr_history = []
    for epoch in range(epochs):
        for batch_idx, (real, _) in enumerate(train_loader):
            real = real.to(device)
            batch_size = real.size(0)

            # --- Critic update ---
            z = torch.randn(batch_size, latent_dim, device=device)
            fake = generator(z).detach()
            critic_loss = critic(fake).mean() - critic(real).mean()
            if variant == "gp":
                critic_loss = critic_loss + gp_lambda * gradient_penalty(critic, real, fake, device)

            opt_c.zero_grad()
            critic_loss.backward()
            opt_c.step()

            if variant == "clip":
                for p in critic.parameters():
                    p.data.clamp_(-clip_value, clip_value)

            # --- Generator update (every n_critic critic steps) ---
            if batch_idx % n_critic == 0:
                z = torch.randn(batch_size, latent_dim, device=device)
                gen_loss = -critic(generator(z)).mean()
                opt_g.zero_grad()
                gen_loss.backward()
                opt_g.step()

        # GANs have no paired reconstruction target; PSNR is intentionally not computed.
        generator.train()
        print(f"[{variant.upper()}] Epoch {epoch+1}/{epochs} | Critic loss: {critic_loss.item():.4f}")

    # Final FID against a held-out test batch
    generator.eval()
    with torch.no_grad():
        real_batch, _ = next(iter(test_loader))
        real_batch = real_batch.to(device)
        z = torch.randn(real_batch.size(0), latent_dim, device=device)
        fake_batch = generator(z)
        final_fid = calculate_colour_frechet_proxy(real_batch, fake_batch)

    return generator, psnr_history, final_fid, time.perf_counter() - run_started


**Discussion — Shared WGAN/WGAN-GP training routine**

- One function drives both variants so the *only* difference between the two runs is the Lipschitz constraint mechanism — weight clipping (`clip_value=0.01`) for WGAN vs. gradient penalty (`gp_lambda=10.0`) for WGAN-GP — isolating that as the variable under comparison, per the assignment's Task 1.
- Critic is updated `n_critic=5` times per generator update, following the original WGAN algorithm, since the critic needs to stay near-optimal for the generator's gradient to approximate the true Wasserstein distance.
- Optimizers differ by design (RMSprop for clipped WGAN, Adam with $\beta_1$=0.5 for WGAN-GP), matching each paper's original recipe rather than reusing one setting for both.
- Per-epoch PSNR here is only a rough sample-fidelity proxy (real vs. randomly generated images, no paired target) — GANs have no encoder, so this should be read as a trend indicator, not a reconstruction metric like the Part A/VQ-VAE PSNR values.

#### Train WGAN and WGAN-GP, Generate Samples, Compare FID

In [ ]:
epochs_gan = 30
latent_dim_gan = 128
gan_results = {}

for variant in ["clip", "gp"]:
    label = "WGAN" if variant == "clip" else "WGAN-GP"
    print(f"\n--- Training {label} ---")
    generator, psnr_history, fid_proxy, train_seconds = train_wgan(variant, epochs=epochs_gan, latent_dim=latent_dim_gan)

    with torch.no_grad():
        z = torch.randn(100, latent_dim_gan, device=device)
        samples = generator(z).cpu()

    gan_results[label] = {
        "generator": generator,
        "psnr_history": psnr_history,
        "colour_frechet_proxy": fid_proxy,
        "train_seconds": train_seconds,
        "samples": samples
    }


**Superseded after the Kubeflow rerun.** Use the standard-Inception-FID comparison at the end of this notebook; GAN PSNR is not a valid evaluation metric.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (label, res) in zip(axes, gan_results.items()):
    grid_img = vutils.make_grid(res["samples"], nrow=10, normalize=True, padding=1)
    ax.imshow(grid_img.permute(1, 2, 0).numpy())
    ax.set_title(f"Random samples: {label}")
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Standard Inception FID is computed in the final evaluation cell below. GAN PSNR is not reported because it is not a reconstruction metric.")


**Superseded after the Kubeflow rerun.** Use the standard-Inception-FID comparison at the end of this notebook; GAN PSNR is not a valid evaluation metric.


## Required standard-FID evaluation and timing

Install once in the Kubeflow kernel if necessary: `!pip install -q "torchmetrics[image]" torch-fidelity`, then restart. The following uses standard pretrained-Inception FID with 5,000 held-out real/generated images.


In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
FID_SAMPLES = 5000
@torch.no_grad()
def standard_inception_fid(generator, n_samples=FID_SAMPLES):
    metric=FrechetInceptionDistance(feature=2048, normalize=True).to(device); seen=0
    for real,_ in test_loader:
        real=real.to(device); take=min(real.size(0), n_samples-seen); metric.update(real[:take], real=True); seen += take
        if seen == n_samples: break
    made=0
    while made<n_samples:
        n=min(128,n_samples-made); metric.update(generator(torch.randn(n, latent_dim_gan, device=device)).clamp(0,1), real=False); made += n
    return float(metric.compute().cpu())

print(f"{'Model':<12}{'Standard FID':>15}{'Train (min)':>15}")
for label, result in gan_results.items():
    result['standard_inception_fid']=standard_inception_fid(result['generator'].eval())
    print(f"{label:<12}{result['standard_inception_fid']:>15.2f}{result['train_seconds']/60:>15.2f}")

Path('part_b_metrics.json').write_text(json.dumps({'fid_samples':FID_SAMPLES, 'gan':{name:{'fid':r['standard_inception_fid'],'train_seconds':r['train_seconds']} for name,r in gan_results.items()}},indent=2))
torch.save({name:r['samples'] for name,r in gan_results.items()}, 'part_b_samples.pt')
print('Saved Part C artifacts: part_b_metrics.json and part_b_samples.pt')
